[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto1/pipeline.ipynb)

# Methodology Extraction Pipeline

Extracts structured methodology from a research paper.

Output format:
```json
{
  "Design": "experiment",
  "Method": ["BERT"],
  "Data": ["MNIST"],
  "Evaluation": ["accuracy"]
}
```

**Run in order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7**

## Setup

In [ ]:
!pip install requests

In [ ]:
import json
import re
from dataclasses import dataclass, field
from enum import Enum

print("Setup complete.")

## Data Models

In [ ]:
class DesignFamily(str, Enum):
    EMPIRICAL = "empirical"
    NON_EMPIRICAL = "non_empirical"
    MIXED = "mixed"
    UNKNOWN = "unknown"


class PrimaryDesignType(str, Enum):
    # Oates empirical strategies
    SURVEY = "survey"
    EXPERIMENT = "experiment"
    CASE_STUDY = "case_study"
    ACTION_RESEARCH = "action_research"
    ETHNOGRAPHY = "ethnography"
    # Oates + Pilkington & Pretorius computing-specific designs
    DESIGN_AND_CREATION = "design_and_creation"
    MODEL_OR_THEORY_BUILDING = "model_or_theory_building"
    UNKNOWN = "unknown"


class DesignSubtype(str, Enum):
    ALGORITHM_DEVELOPMENT = "algorithm_development"
    SYSTEM_DEVELOPMENT = "system_development"
    MODEL_BUILDING = "model_building"
    THEORY_BUILDING = "theory_building"
    NONE = "none"


@dataclass
class ResearchDesign:
    family: DesignFamily = DesignFamily.UNKNOWN
    primary_type: PrimaryDesignType = PrimaryDesignType.UNKNOWN
    subtype: DesignSubtype = DesignSubtype.NONE
    secondary_types: list = field(default_factory=list)

    def to_dict(self):
        return {
            "family": self.family.value,
            "primary_type": self.primary_type.value,
            "subtype": self.subtype.value,
            "secondary_types": [t.value for t in self.secondary_types],
        }


@dataclass
class CandidateWithContext:
    candidate: str
    sentence: str
    section: str = "unknown"
    source_paper: str = ""


@dataclass
class MethodologyProfile:
    design: ResearchDesign = field(default_factory=ResearchDesign)
    technical_method: list = field(default_factory=list)
    task: list = field(default_factory=list)
    data: list = field(default_factory=list)
    evaluation: list = field(default_factory=list)

    def to_dict(self):
        result = {
            "ResearchDesign": self.design.to_dict(),
            "TechnicalMethod": self.technical_method,
            "Task": self.task,
        }
        optional = {}
        if self.data:
            optional["Data"] = self.data
        if self.evaluation:
            optional["Evaluation"] = self.evaluation
        if optional:
            result["Optional"] = optional
        return result


print("Models ready.")

## Step 0 — Load TEI XML

Upload a TEI XML file produced by local GROBID (`python pdf_to_xml.py paper.pdf`).

In [ ]:
from xml.etree import ElementTree as ET

from google.colab import files

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
SKIP_HEADINGS = {"references", "acknowledgements", "acknowledgments"}


def _text(element) -> str:
    return " ".join(element.itertext()).strip()


uploaded = files.upload()
xml_filename = next(iter(uploaded))
xml_bytes = uploaded[xml_filename]

root = ET.fromstring(xml_bytes.decode("utf-8"))

abstract_el = root.find(".//tei:abstract", NS)
abstract_text = _text(abstract_el) if abstract_el is not None else ""

sections = []
for div in root.findall(".//tei:body//tei:div", NS):
    heading = div.findtext("tei:head", namespaces=NS) or ""
    if heading.lower().strip() in SKIP_HEADINGS:
        continue
    body = " ".join(_text(p) for p in div.findall("tei:p", NS)).strip()
    if body:
        sections.append({"heading": heading, "text": body})

if abstract_text:
    sections.insert(0, {"heading": "Abstract", "text": abstract_text})

print(f"Loaded : {xml_filename}")
print(f"Sections: {len(sections)}")
for s in sections:
    print(f"  - {s['heading']}")

## Step 1 — Candidate Extraction (SciBERT NER)

In [ ]:
_STOP = {
    "We",
    "Our",
    "The",
    "This",
    "In",
    "A",
    "An",
    "To",
    "For",
    "On",
    "Is",
    "It",
    "At",
    "By",
    "As",
    "Of",
    "Be",
    "Are",
    "Was",
    "Has",
    "Have",
    "From",
    "With",
    "That",
    "Which",
    "These",
    "Those",
    "Also",
    "Such",
    "Both",
    "Each",
}

_PATTERNS: list[tuple[str, int]] = [
    (r"\b[A-Z][A-Za-z0-9]*(?:-[A-Za-z0-9]+)*\b", 0),
    (r"\b[a-z]+-\d+\b", 0),
    (
        r"\b(?:accuracy|f1|precision|recall|bleu|rouge|auc"
        r"|attention|translation|summarization|classification|recognition|detection|generation)\b",
        re.IGNORECASE,
    ),
    (r"\b(?:dataset|corpus|benchmark)\b", re.IGNORECASE),
]


def extract_candidates(sections: list[dict]) -> list[CandidateWithContext]:
    results: list[CandidateWithContext] = []
    seen: set[str] = set()
    for section in sections:
        heading = section["heading"]
        sentences = [
            s.strip() for s in re.split(r"(?<=[.!?])\s+", section["text"]) if s.strip()
        ]
        for sent in sentences:
            for pattern, flags in _PATTERNS:
                for match in re.finditer(pattern, sent, flags):
                    term = match.group().strip()
                    if (
                        len(term) < 3
                        or len(term) > 40
                        or term in _STOP
                        or term.lower() in {s.lower() for s in _STOP}
                        or term in seen
                    ):
                        continue
                    seen.add(term)
                    results.append(
                        CandidateWithContext(
                            candidate=term, sentence=sent, section=heading
                        )
                    )
    return results


candidates_with_ctx = extract_candidates(sections)
print(f"Candidates: {len(candidates_with_ctx)}")

## Step 2 — Role Classification

In [ ]:
class Role(str, Enum):
    TECHNICAL_METHOD = "TechnicalMethod"
    TASK = "Task"
    DATA = "Data"
    EVALUATION = "Evaluation"
    OTHER = "Other"


_KNOWN_METHODS = {
    "bert",
    "gpt",
    "roberta",
    "xlnet",
    "t5",
    "gpt-2",
    "gpt-3",
    "lstm",
    "cnn",
    "rnn",
    "transformer",
    "attention",
    "svm",
    "random forest",
    "k-means",
    "resnet",
    "vgg",
    "scibert",
}
_KNOWN_TASKS = {
    "question answering",
    "text classification",
    "image classification",
    "named entity recognition",
    "machine translation",
    "summarization",
    "sentiment analysis",
    "relation extraction",
    "coreference resolution",
}
_KNOWN_DATA = {
    "mnist",
    "cifar",
    "squad",
    "glue",
    "superglue",
    "imagenet",
    "sst-2",
    "sst-1",
    "conll",
    "wikitext",
    "bookcorpus",
    "imdb",
    "yelp",
    "amazon",
    "snli",
    "mnli",
}
_KNOWN_EVAL = {
    "accuracy",
    "f1",
    "precision",
    "recall",
    "bleu",
    "rouge",
    "auc",
    "map",
    "ndcg",
    "perplexity",
    "em",
    "exact match",
}

_METHOD_RE = re.compile(
    r"\b(neural|network|model|algorithm|architecture|classifier|proposed)\b",
    re.IGNORECASE,
)
_TASK_RE = re.compile(
    r"\b(task|problem|classification|recognition|detection|generation|translation)\b",
    re.IGNORECASE,
)
_DATA_RE = re.compile(
    r"\b(dataset|corpus|benchmark|collection|training|test)\b",
    re.IGNORECASE,
)
_EVAL_RE = re.compile(
    r"\b(score|metric|performance|rate|result)\b",
    re.IGNORECASE,
)


def classify_role(candidate: str, context: str = "") -> Role:
    low = candidate.lower()
    if low in _KNOWN_EVAL or _EVAL_RE.search(low):
        return Role.EVALUATION
    if low in _KNOWN_DATA or _DATA_RE.search(context.lower()):
        return Role.DATA
    if low in _KNOWN_TASKS or _TASK_RE.search(context.lower()):
        return Role.TASK
    if low in _KNOWN_METHODS or _METHOD_RE.search(context.lower()):
        return Role.TECHNICAL_METHOD
    return Role.OTHER


classified: dict[str, Role] = {}
for cwc in candidates_with_ctx:
    classified[cwc.candidate] = classify_role(cwc.candidate, cwc.sentence)

for term, role in sorted(classified.items()):
    print(f"  {role.value:16s}  {term}")

## Candidate Log (Error Analysis)


In [ ]:
log = []
for cwc in candidates_with_ctx:
    log.append(
        {
            "candidate": cwc.candidate,
            "role": classified.get(cwc.candidate, Role.OTHER).value,
            "sentence": cwc.sentence,
            "section": cwc.section,
        }
    )

print(json.dumps(log, indent=2))

## Step 3 — Design Detection

In [ ]:
_NON_EMPIRICAL = {
    PrimaryDesignType.DESIGN_AND_CREATION,
    PrimaryDesignType.MODEL_OR_THEORY_BUILDING,
}

_PRIMARY_PATTERNS: list[tuple[PrimaryDesignType, list[str]]] = [
    (
        PrimaryDesignType.SURVEY,
        [r"\bsurvey\b", r"\bquestionnaire\b", r"\bliterature review\b"],
    ),
    (
        PrimaryDesignType.EXPERIMENT,
        [r"\bexperiment\w*\b", r"\bablation\b", r"\buser study\b"],
    ),
    (PrimaryDesignType.CASE_STUDY, [r"\bcase study\b", r"\bcase studies\b"]),
    (PrimaryDesignType.ACTION_RESEARCH, [r"\baction research\b", r"\bparticipatory\b"]),
    (PrimaryDesignType.ETHNOGRAPHY, [r"\bethnograph\w*\b", r"\bfieldwork\b"]),
    (
        PrimaryDesignType.DESIGN_AND_CREATION,
        [
            r"\bpropose\w*\b",
            r"\bnovel\b",
            r"\barchitecture\b",
            r"\bartefact\b",
            r"\bartifact\b",
        ],
    ),
    (
        PrimaryDesignType.MODEL_OR_THEORY_BUILDING,
        [r"\btheor\w+\b", r"\bformal\w*\b", r"\bproof\b"],
    ),
]

_SUBTYPE_PATTERNS: list[tuple[DesignSubtype, list[str]]] = [
    (DesignSubtype.ALGORITHM_DEVELOPMENT, [r"\balgorithm\w*\b", r"\boptimiz\w*\b"]),
    (
        DesignSubtype.SYSTEM_DEVELOPMENT,
        [r"\bsystem\b", r"\binfrastructure\b", r"\bpipeline\b"],
    ),
    (DesignSubtype.MODEL_BUILDING, [r"\bmodel\b", r"\bneural\b", r"\bnetwork\b"]),
    (DesignSubtype.THEORY_BUILDING, [r"\btheor\w+\b", r"\blemma\b", r"\bproof\b"]),
]


def detect_design(text: str) -> ResearchDesign:
    text_lower = text.lower()

    primary_scores: dict[PrimaryDesignType, int] = {}
    for ptype, patterns in _PRIMARY_PATTERNS:
        count = sum(len(re.findall(p, text_lower)) for p in patterns)
        if count > 0:
            primary_scores[ptype] = count

    if not primary_scores:
        return ResearchDesign()

    primary_type = max(primary_scores, key=lambda k: primary_scores[k])

    if primary_type in _NON_EMPIRICAL:
        exp_score = primary_scores.get(PrimaryDesignType.EXPERIMENT, 0)
        if exp_score > 0:
            family = DesignFamily.MIXED
            secondary_types = [PrimaryDesignType.EXPERIMENT]
        else:
            family = DesignFamily.NON_EMPIRICAL
            secondary_types = []
    else:
        family = DesignFamily.EMPIRICAL
        secondary_types = []

    subtype = DesignSubtype.NONE
    if primary_type == PrimaryDesignType.DESIGN_AND_CREATION:
        subtype_scores: dict[DesignSubtype, int] = {}
        for stype, patterns in _SUBTYPE_PATTERNS:
            count = sum(len(re.findall(p, text_lower)) for p in patterns)
            if count > 0:
                subtype_scores[stype] = count
        if subtype_scores:
            subtype = max(subtype_scores, key=lambda k: subtype_scores[k])

    return ResearchDesign(
        family=family,
        primary_type=primary_type,
        subtype=subtype,
        secondary_types=secondary_types,
    )


full_text = " ".join(s["text"] for s in sections)
design = detect_design(full_text)
print(json.dumps(design.to_dict(), indent=2))

## Step 4 — Build JSON Output

In [ ]:
profile = MethodologyProfile(
    design=design,
    technical_method=[t for t, r in classified.items() if r == Role.TECHNICAL_METHOD],
    task=[t for t, r in classified.items() if r == Role.TASK],
    data=[t for t, r in classified.items() if r == Role.DATA],
    evaluation=[t for t, r in classified.items() if r == Role.EVALUATION],
)

print(json.dumps(profile.to_dict(), indent=2))

## Step 5 — Consistency Checking

In [ ]:
@dataclass
class ConsistencyResult:
    is_valid: bool
    warnings: list


def check_consistency(profile: MethodologyProfile) -> ConsistencyResult:
    warnings = []
    needs_check = (
        profile.design.primary_type
        in (PrimaryDesignType.EXPERIMENT, PrimaryDesignType.DESIGN_AND_CREATION)
        or PrimaryDesignType.EXPERIMENT in profile.design.secondary_types
    )
    if needs_check:
        if not profile.task:
            warnings.append("Experimental paper without Task is weak.")
        if not profile.technical_method:
            warnings.append("Experimental paper without TechnicalMethod is weak.")
    if profile.technical_method and not profile.task:
        warnings.append("TechnicalMethod without Task may be incomplete.")
    return ConsistencyResult(is_valid=len(warnings) == 0, warnings=warnings)


result = check_consistency(profile)
print(f"Valid: {result.is_valid}")
for w in result.warnings:
    print(f"  WARNING: {w}")
if result.is_valid:
    print("  No issues found.")